In [ ]:
#Importing Libraries
import ssl #Data security of MQTT messages
import json #Convert the Python data into JSON format
import time #Time period for message flow
import paho.mqtt.client as mqtt #MQTT Library

#HiveMQ Cluster Credentials for ESP 8266
ESP_BROKER = "8644400fc9e448468233fc0f9be265a0.s1.eu.hivemq.cloud"
ESP_PORT = 8883
ESP_USERNAME = "ParqHome_Hardware"
ESP_PASSWORD = "ParqHome_Hardware@123"

#HiveMQ Cluster Credentials for User Interface
UI_BROKER = "4e3fe38828204ed2887d34fe15744691.s1.eu.hivemq.cloud"
UI_PORT = 8883
UI_USERNAME = "ParqHome_Software"
UI_PASSWORD = "ParqHome_Software@123"

#ESP 8266 Topics
ESP_TELEMETRY_TOPIC = "home/esp/status"
ESP_COMMAND_TOPIC = "home/esp/control"

#User Interface Topics
UI_DATA_TOPIC = "home/ui/status"
UI_COMMAND_TOPIC = "home/ui/control"

#MQTT Clients
esp_client = mqtt.Client(client_id="ESP_Client")
ui_client = mqtt.Client(client_id="UI_Client")

#TLS SECURITY CONNECTION
def setup_tls(client):

    client.tls_set(
        cert_reqs=ssl.CERT_REQUIRED,
        tls_version=ssl.PROTOCOL_TLS
    )

    client.tls_insecure_set(False)

#CONNECTION CALLBACKS
#ESP CONNECTIONS
def on_connect_esp(client, userdata, flags, rc):

    print("ESP Connected")

    client.subscribe(ESP_TELEMETRY_TOPIC)

#UI CONNECTIONS
def on_connect_ui(client, userdata, flags, rc):

    print("UI Connected")

    client.subscribe(UI_COMMAND_TOPIC)

#Receiving messages from ESP
def on_message_esp(client, userdata, msg):

    payload = msg.payload.decode() #Payload decoded

    print("\nESP SENT:")
    print(payload)

    try:

        data = json.loads(payload) #Convert JSON to Python dictionary
        #Forward data to UI
        ui_client.publish(
            UI_DATA_TOPIC,
            json.dumps(data),
            qos=1
        )

    except Exception as e:

        print("ESP Error:", e)

#Receiving message from UI
def on_message_ui(client, userdata, msg):

    payload = msg.payload.decode()

    print("\nUI SENT:")
    print(payload)

    try:

        data = json.loads(payload)
        #Forward Data to ESP
        esp_client.publish(
            ESP_COMMAND_TOPIC,
            json.dumps(data),
            qos=1
        )

    except Exception as e:

        print("UI Error:", e)

#Authentication
esp_client.username_pw_set(
    ESP_USERNAME,
    ESP_PASSWORD
)

ui_client.username_pw_set(
    UI_USERNAME,
    UI_PASSWORD
)


setup_tls(esp_client)
setup_tls(ui_client)

#Attaching callbacks
esp_client.on_connect = on_connect_esp
esp_client.on_message = on_message_esp

ui_client.on_connect = on_connect_ui
ui_client.on_message = on_message_ui

#Connecting to brokers
esp_client.connect(
    ESP_BROKER,
    ESP_PORT,
    60
)

ui_client.connect(
    UI_BROKER,
    UI_PORT,
    60
)

#Starting MQTT Network Loops
esp_client.loop_start()
ui_client.loop_start()


print("\nMQTT Bridge Running...\n")

#Keeping the program running
while True:

    time.sleep(1)

C:\Users\91984\AppData\Local\Temp\ipykernel_19616\2192882071.py:29: DeprecationWarning: Callback API version 1 is deprecated, update to latest version
  esp_client = mqtt.Client(client_id="ESP_Client")
C:\Users\91984\AppData\Local\Temp\ipykernel_19616\2192882071.py:30: DeprecationWarning: Callback API version 1 is deprecated, update to latest version
  ui_client = mqtt.Client(client_id="UI_Client")


ESP Connected

MQTT Bridge Running...


ESP SENT:
{"source":"device","familyId":"ZUH-812280","switchId":"003","status":false}
UI Connected

ESP SENT:
{"source":"device","familyId":"ZUH-812280","switchId":"001","status":true}

ESP SENT:
{"source":"device","familyId":"ZUH-812280","switchId":"003","status":true}

ESP SENT:
{"source":"device","familyId":"ZUH-812280","switchId":"002","status":true}

ESP SENT:
{"source":"device","familyId":"ZUH-812280","switchId":"002","status":false}

ESP SENT:
{"source":"device","familyId":"ZUH-812280","switchId":"001","status":false}

ESP SENT:
{"source":"device","familyId":"ZUH-812280","switchId":"003","status":false}

UI SENT:
{"source":"app","familyId":"ZUH-812280","switchId":"001","status":true}

UI SENT:
{"source":"app","familyId":"ZUH-812280","switchId":"003","status":true}

UI SENT:
{"source":"app","familyId":"ZUH-812280","switchId":"002","status":true,"fanSpeed":5}

UI SENT:
{"source":"app","familyId":"ZUH-812280","switchId":"001","status":false}
